# Labsense SQL Plots Between Two Timestamps

This notebook loads SQL data between `START_TS` and `END_TS` and reuses existing Labsense dashboard plotting functions.

Notes:
- It uses `Labsense_SQL.sql_helpers` for SQL environment and connection string setup.
- Existing dashboard plotting functions include their own internal "recent window" filters:
  - Electricity: last two years for daily trends and previous day for one subplot.
  - Water: last two months.
  - Fumehood: last 7 days.
- To see all expected plots, choose a timestamp window that overlaps those periods.

In [4]:
from pathlib import Path
import os
import sys

import pandas as pd
import pyodbc
from dotenv import load_dotenv
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    """Walk upward until the folder containing Labsense_SQL is found."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "Labsense_SQL").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root containing Labsense_SQL")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from Labsense_SQL import sql_helpers
from Labsense_SQL.consumption_dashboard import (
    CALCULATE_IDLE_POWER,
    calculate_idle_power,
    create_plots as create_consumption_plots,
)
from Labsense_SQL.Fumehood_dashboard import create_plots as create_fumehood_plots
from Labsense_SQL.presence_utils import get_room_light_presence_data
from Labsense_SQL.water_dashboard import create_plots as create_water_plots

# Force-load the .env from Labsense_SQL
env_path = repo_root / "Labsense_SQL" / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path, override=True)
else:
    print(f"Warning: .env not found at {env_path}")

# Keep helper behavior as well (safe no-op if already loaded)
sql_helpers._load_sql_env()


def build_connection_string_with_driver_fallback() -> str:
    """Build SQL connection string using installed ODBC drivers."""
    sql_server = os.getenv("SQL_SERVER", "MSM-FPM-70203\\LABSENSE").strip()
    sql_database = os.getenv("SQL_DATABASE", "labsense").strip()
    sql_trusted = os.getenv("SQL_TRUSTED_CONNECTION", "yes").strip().lower()
    sql_encrypt = os.getenv("SQL_ENCRYPTION", "Optional").strip()
    sql_user = os.getenv("SQL_USER", "").strip()
    sql_password = os.getenv("SQL_PASSWORD", "").strip()

    available_drivers = pyodbc.drivers()
    preferred_driver = os.getenv("SQL_DRIVER", "ODBC Driver 18 for SQL Server").strip()
    driver_candidates = [
        preferred_driver,
        "ODBC Driver 18 for SQL Server",
        "ODBC Driver 17 for SQL Server",
        "SQL Server",
    ]
    selected_driver = next((d for d in driver_candidates if d in available_drivers), None)

    if selected_driver is None:
        if not available_drivers:
            raise RuntimeError("No ODBC drivers found. Install SQL Server ODBC driver.")
        selected_driver = available_drivers[0]

    if sql_trusted == "yes":
        conn = (
            f"DRIVER={{{selected_driver}}};"
            f"SERVER={sql_server};"
            f"DATABASE={sql_database};"
            "Trusted_Connection=yes;"
            f"Encrypt={sql_encrypt}"
        )
    else:
        if not sql_user or not sql_password:
            raise ValueError(
                "SQL_USER and SQL_PASSWORD must be set when SQL_TRUSTED_CONNECTION is not 'yes'."
            )
        conn = (
            f"DRIVER={{{selected_driver}}};"
            f"SERVER={sql_server};"
            f"DATABASE={sql_database};"
            f"UID={sql_user};"
            f"PWD={sql_password};"
            f"Encrypt={sql_encrypt}"
        )

    print(f"Loaded .env: {env_path}")
    print(f"Using ODBC driver: {selected_driver}")
    print(f"SQL server/database: {sql_server} / {sql_database}")
    return conn


connection_string = build_connection_string_with_driver_fallback()

# --- User parameters ---
START_TS = pd.Timestamp("2026-05-11 00:00:00")
END_TS = pd.Timestamp("2026-05-15 23:59:59")
PRESENCE_LAB_ID = 1
PRESENCE_SUBLAB_ID = 3

if END_TS <= START_TS:
    raise ValueError("END_TS must be later than START_TS")

PLOT_DIR = repo_root / "plots" / f"notebook_{START_TS:%Y%m%d_%H%M}_{END_TS:%Y%m%d_%H%M}"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using SQL window: {START_TS} -> {END_TS}")
print(f"Plot output dir: {PLOT_DIR}")

Loaded .env: C:\Users\takas\src\labsense\Labsense_SQL\.env
Using ODBC driver: SQL Server
SQL server/database: MSM-FPM-70203\LABSENSE / labsense
Using SQL window: 2026-05-11 00:00:00 -> 2026-05-15 23:59:59
Plot output dir: C:\Users\takas\src\labsense\plots\notebook_20260511_0000_20260515_2359


In [ ]:
def read_sql_df(connection, query: str, start_ts: pd.Timestamp, end_ts: pd.Timestamp) -> pd.DataFrame:
    """Run a bounded SQL query and coerce Timestamp/Datestamp columns to datetime."""
    df = pd.read_sql(
        query,
        connection,
        params=[start_ts.to_pydatetime(), end_ts.to_pydatetime()],
    )
    if "Timestamp" in df.columns:
        df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    if "Datestamp" in df.columns:
        df["Datestamp"] = pd.to_datetime(df["Datestamp"])
    return df


with pyodbc.connect(connection_string) as conn:
    df_elec_daily = read_sql_df(
        conn,
        """
        SELECT id, Esum, Datestamp
        FROM dbo.elecDaily
        WHERE Datestamp >= ? AND Datestamp <= ?
        ORDER BY Datestamp ASC
        """,
        START_TS,
        END_TS,
    )

    df_elec_minute = read_sql_df(
        conn,
        """
        SELECT id, EnergyValue, Timestamp
        FROM dbo.elecMinute
        WHERE Timestamp >= ? AND Timestamp <= ?
        ORDER BY Timestamp ASC
        """,
        START_TS,
        END_TS,
    )

    df_fumehood = read_sql_df(
        conn,
        """
        SELECT id, LabId, SubLabId, Distance, Light, Airflow, Timestamp
        FROM dbo.fumehood
        WHERE Timestamp >= ? AND Timestamp <= ?
        ORDER BY Timestamp ASC
        """,
        START_TS,
        END_TS,
    )

    df_water = read_sql_df(
        conn,
        """
        SELECT id, LabId, SublabId, Water, Timestamp
        FROM dbo.water
        WHERE Timestamp >= ? AND Timestamp <= ?
        ORDER BY Timestamp ASC
        """,
        START_TS,
        END_TS,
    )

# Presence series used by electricity previous-day subplot logic
light_for_presence = df_fumehood[
    (df_fumehood["LabId"] == PRESENCE_LAB_ID)
    & (df_fumehood["SubLabId"] == PRESENCE_SUBLAB_ID)
][["Light", "Timestamp"]].copy()

presence_df = get_room_light_presence_data(
    light_for_presence,
    lab_id=PRESENCE_LAB_ID,
    sublab_id=PRESENCE_SUBLAB_ID,
)

display(
    pd.DataFrame(
        {
            "dataset": ["elecDaily", "elecMinute", "fumehood", "water", "presence"],
            "rows": [
                len(df_elec_daily),
                len(df_elec_minute),
                len(df_fumehood),
                len(df_water),
                0 if presence_df is None else len(presence_df),
            ],
        }
    )
)

In [ ]:
idle_power_kw = 0.0
if CALCULATE_IDLE_POWER and not df_elec_minute.empty:
    idle_power_kw = calculate_idle_power(df_elec_minute)

consumption_plot_files = create_consumption_plots(
    df=df_elec_daily,
    plot_dir=PLOT_DIR,
    df_granular=df_elec_minute,
    idle_power_kw=idle_power_kw,
    presence_df=presence_df,
)

water_plot_files = create_water_plots(
    df=df_water,
    plot_dir=PLOT_DIR,
)

fumehood_plot_files = create_fumehood_plots(
    df=df_fumehood,
    plot_dir=PLOT_DIR,
)

print("Consumption plots:", sorted(consumption_plot_files.keys()))
print("Water plots:", len(water_plot_files))
print("Fumehood plots:", len(fumehood_plot_files))

In [ ]:
generated_pngs = sorted(PLOT_DIR.glob("*.png"))
summary_df = pd.DataFrame(
    {
        "plot_file": [p.name for p in generated_pngs],
        "size_kb": [round(p.stat().st_size / 1024, 1) for p in generated_pngs],
    }
)

display(summary_df)

if not summary_df.empty:
    print(f"Generated {len(summary_df)} plot image(s) in {PLOT_DIR}")
else:
    print("No plots were generated. Verify timestamp window and dashboard internal lookback windows.")